# SBOM & Vulnerability Analysis Dashboard

This notebook reads the JSON files generated by **Syft** (SBOMs) and **Grype** (vulnerability reports) from the `/data` directory and performs quantitative analysis of the organization's security posture.

---

## Analyses Included
1. **Vulnerability Distribution by Severity** - Pie chart of Critical / High / Medium / Low / Negligible
2. **Top 3 Most Insecure Repositories** - Repos with the highest vulnerability concentration
3. **Top 5 Guilty Dependencies** - Packages introducing the most vulnerabilities (prioritizing Critical & High)

## Setup & Data Loading

In [ ]:
import json
import glob
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.2)
plt.rcParams.update({
    "figure.figsize": (12, 7),
    "figure.dpi": 100,
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
})

SEVERITY_COLORS = {
    "Critical": "#DC2626",
    "High":     "#EA580C",
    "Medium":   "#D97706",
    "Low":      "#65A30D",
    "Negligible": "#6B7280",
    "Unknown":  "#9CA3AF",
}

SEVERITY_ORDER = ["Critical", "High", "Medium", "Low", "Negligible", "Unknown"]

DATA_DIR = "/data"
OUTPUT_DIR = "/home/jovyan/work"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Files found: {len(os.listdir(DATA_DIR))}")

In [ ]:
def load_vulnerability_data(data_dir: str) -> pd.DataFrame:
    """Parse all Grype JSON files into a flat DataFrame (one row per vulnerability match)."""
    records = []
    vuln_files = sorted(glob.glob(os.path.join(data_dir, "vuln_*.json")))

    if not vuln_files:
        print("No vulnerability files found in", data_dir)
        return pd.DataFrame()

    print(f"Loading {len(vuln_files)} vulnerability reports...")

    for filepath in vuln_files:
        filename = Path(filepath).stem
        repo_name = filename.replace("vuln_", "", 1)

        try:
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
        except (json.JSONDecodeError, IOError) as e:
            print(f"  Skipping {filename}: {e}")
            continue

        for match in data.get("matches", []):
            vulnerability = match.get("vulnerability", {})
            artifact = match.get("artifact", {})

            records.append({
                "repository": repo_name,
                "vuln_id": vulnerability.get("id", "N/A"),
                "severity": vulnerability.get("severity", "Unknown"),
                "description": vulnerability.get("description", "")[:200],
                "data_source": vulnerability.get("dataSource", ""),
                "package_name": artifact.get("name", "Unknown"),
                "package_version": artifact.get("version", "N/A"),
                "package_type": artifact.get("type", "N/A"),
                "package_language": artifact.get("language", "N/A"),
            })

    df = pd.DataFrame(records)

    if not df.empty:
        df["severity"] = df["severity"].str.capitalize()
        df["severity"] = pd.Categorical(
            df["severity"], categories=SEVERITY_ORDER, ordered=True
        )

    print(f"Loaded {len(df)} vulnerability records from {len(vuln_files)} repositories.")
    return df


df_vulns = load_vulnerability_data(DATA_DIR)
df_vulns.head(10)

In [ ]:
manifest_path = os.path.join(DATA_DIR, "manifest.json")
if os.path.exists(manifest_path):
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    print(f"Organization:          {manifest.get('organization', 'N/A')}")
    print(f"Scan date:             {manifest.get('generated_at', 'N/A')}")
    print(f"Repos processed:       {manifest.get('total_repos_processed', 0)}")
    print(f"SBOMs generated:       {manifest.get('successful_sboms', 0)}")
    print(f"Vuln reports generated: {manifest.get('successful_vuln_reports', 0)}")
    print(f"Errors:                {manifest.get('errors', 0)}")
    print(f"Duration:              {manifest.get('processing_duration_seconds', 0):.1f}s")
else:
    print("manifest.json not found. The processor may still be running.")

---

## 1. Vulnerability Distribution by Severity

Pie chart showing the proportion of total vulnerabilities across severity levels.

In [ ]:
if df_vulns.empty:
    print("No data available for analysis.")
else:
    severity_counts = df_vulns["severity"].value_counts().sort_index()
    severity_counts = severity_counts[severity_counts > 0]
    colors = [SEVERITY_COLORS.get(sev, "#9CA3AF") for sev in severity_counts.index]

    fig, ax = plt.subplots(figsize=(10, 8))

    wedges, texts, autotexts = ax.pie(
        severity_counts.values,
        labels=severity_counts.index,
        colors=colors,
        autopct=lambda pct: f"{pct:.1f}%\n({int(round(pct / 100.0 * severity_counts.sum()))})",
        startangle=140,
        pctdistance=0.75,
        wedgeprops={"linewidth": 2, "edgecolor": "white"},
        textprops={"fontsize": 12},
    )

    for autotext in autotexts:
        autotext.set_fontsize(10)
        autotext.set_fontweight("bold")

    ax.set_title(
        f"Vulnerability Distribution by Severity\n(Total: {severity_counts.sum():,} vulnerabilities)",
        fontsize=16, fontweight="bold", pad=20,
    )

    plt.tight_layout()
    chart_path = os.path.join(OUTPUT_DIR, "chart_severity_distribution.png")
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nChart saved to {chart_path}")

    print("\n-- Severity Summary Table --")
    summary_df = severity_counts.reset_index()
    summary_df.columns = ["Severity", "Count"]
    summary_df["Percentage"] = (summary_df["Count"] / summary_df["Count"].sum() * 100).round(1)
    display(summary_df)

---

## 2. Top 3 Most Insecure Repositories

Identifies the repositories with the highest concentration of vulnerabilities, broken down by severity.

In [ ]:
if df_vulns.empty:
    print("No data available for analysis.")
else:
    repo_severity = df_vulns.groupby(["repository", "severity"]).size().unstack(fill_value=0)
    repo_severity["Total"] = repo_severity.sum(axis=1)

    # Weighted risk score: Critical=4, High=3, Medium=2, Low=1
    weights = {"Critical": 4, "High": 3, "Medium": 2, "Low": 1, "Negligible": 0.5, "Unknown": 0.5}
    repo_severity["Risk Score"] = sum(
        repo_severity.get(sev, 0) * w for sev, w in weights.items()
    )

    top3 = repo_severity.sort_values("Risk Score", ascending=False).head(3)

    print("-- Top 3 Most Insecure Repositories --\n")
    display(top3)

    # Horizontal stacked bar chart
    severity_cols = [col for col in SEVERITY_ORDER if col in top3.columns]
    plot_data = top3[severity_cols].iloc[::-1]

    fig, ax = plt.subplots(figsize=(12, 5))
    left = pd.Series([0] * len(plot_data), index=plot_data.index)

    for sev in severity_cols:
        values = plot_data[sev]
        color = SEVERITY_COLORS.get(sev, "#9CA3AF")
        bars = ax.barh(plot_data.index, values, left=left, label=sev, color=color, edgecolor="white", linewidth=0.5)
        left += values

        for bar, val in zip(bars, values):
            if val > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_y() + bar.get_height() / 2,
                    str(int(val)),
                    ha="center", va="center",
                    fontsize=10, fontweight="bold", color="white",
                )

    ax.set_xlabel("Number of Vulnerabilities", fontsize=12)
    ax.set_title("Top 3 Most Insecure Repositories (by Risk Score)", fontsize=16, fontweight="bold", pad=15)
    ax.legend(title="Severity", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    chart_path = os.path.join(OUTPUT_DIR, "chart_top3_repos.png")
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nChart saved to {chart_path}")

---

## 3. Top 5 Guilty Dependencies

Identifies the 5 specific packages/libraries introducing the most vulnerabilities across the entire organization, prioritizing high and critical severity.

In [ ]:
if df_vulns.empty:
    print("No data available for analysis.")
else:
    pkg_vulns = df_vulns.groupby(["package_name", "package_version", "package_type"]).agg(
        total_vulns=("vuln_id", "count"),
        critical=("severity", lambda x: (x == "Critical").sum()),
        high=("severity", lambda x: (x == "High").sum()),
        medium=("severity", lambda x: (x == "Medium").sum()),
        low=("severity", lambda x: (x == "Low").sum()),
        repos_affected=("repository", "nunique"),
        unique_cves=("vuln_id", "nunique"),
    ).reset_index()

    pkg_vulns["risk_score"] = (
        pkg_vulns["critical"] * 4 +
        pkg_vulns["high"] * 3 +
        pkg_vulns["medium"] * 2 +
        pkg_vulns["low"] * 1
    )

    pkg_vulns = pkg_vulns.sort_values(
        ["risk_score", "total_vulns"], ascending=False
    )

    top5 = pkg_vulns.head(5).copy()

    display_df = top5[[
        "package_name", "package_version", "package_type",
        "total_vulns", "critical", "high", "medium", "low",
        "repos_affected", "unique_cves", "risk_score",
    ]].copy()

    display_df.columns = [
        "Package", "Version", "Type",
        "Total Vulns", "Critical", "High", "Medium", "Low",
        "Repos Affected", "Unique CVEs", "Risk Score",
    ]

    display_df = display_df.reset_index(drop=True)
    display_df.index = display_df.index + 1
    display_df.index.name = "Rank"

    print("-- Top 5 Guilty Dependencies --")
    print("(Ranked by weighted risk score: Critical x4, High x3, Medium x2, Low x1)\n")

    def highlight_severity(val, col_name):
        if col_name == "Critical" and val > 0:
            return "background-color: #FEE2E2; color: #DC2626; font-weight: bold"
        elif col_name == "High" and val > 0:
            return "background-color: #FFEDD5; color: #EA580C; font-weight: bold"
        return ""

    styled = display_df.style.map(
        lambda v: highlight_severity(v, "Critical"), subset=["Critical"]
    ).map(
        lambda v: highlight_severity(v, "High"), subset=["High"]
    ).set_properties(**{
        "text-align": "center",
    }).set_properties(
        subset=["Package"], **{"text-align": "left", "font-weight": "bold"}
    )

    display(styled)

In [ ]:
if not df_vulns.empty and not top5.empty:
    fig, ax = plt.subplots(figsize=(14, 6))

    plot_pkgs = top5["package_name"].values
    x = range(len(plot_pkgs))
    bar_width = 0.2

    for i, (sev, color) in enumerate([
        ("critical", SEVERITY_COLORS["Critical"]),
        ("high", SEVERITY_COLORS["High"]),
        ("medium", SEVERITY_COLORS["Medium"]),
        ("low", SEVERITY_COLORS["Low"]),
    ]):
        offset = (i - 1.5) * bar_width
        values = top5[sev].values
        bars = ax.bar(
            [xi + offset for xi in x], values,
            width=bar_width, label=sev.capitalize(), color=color,
            edgecolor="white", linewidth=0.5,
        )
        for bar, val in zip(bars, values):
            if val > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                    str(int(val)), ha="center", va="bottom", fontsize=9, fontweight="bold",
                )

    ax.set_xticks(list(x))
    ax.set_xticklabels(plot_pkgs, rotation=30, ha="right", fontsize=11)
    ax.set_ylabel("Number of Vulnerabilities", fontsize=12)
    ax.set_title("Top 5 Guilty Dependencies - Vulnerability Breakdown", fontsize=16, fontweight="bold", pad=15)
    ax.legend(title="Severity", fontsize=10)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    chart_path = os.path.join(OUTPUT_DIR, "chart_top5_packages.png")
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nChart saved to {chart_path}")

---

## Full Vulnerability Data Export

Export the complete dataset to CSV for further analysis.

In [ ]:
if not df_vulns.empty:
    csv_path = os.path.join(OUTPUT_DIR, "all_vulnerabilities.csv")
    df_vulns.to_csv(csv_path, index=False)
    print(f"Full dataset exported to {csv_path}")
    print(f"   Total records: {len(df_vulns):,}")
    print(f"   Columns: {', '.join(df_vulns.columns)}")
else:
    print("No data to export.")